In [1]:
# Imports
import sys
from pathlib import Path

# Resolve project root and ensure it's on sys.path
ROOT = Path.cwd().resolve()
for _ in range(5):
    if (ROOT / "pyproject.toml").exists() or (ROOT / "raw_data").exists():
        break
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import discretize_preprocess

In [2]:
# Preprocess data
from pathlib import Path

dataset_path = ROOT / "raw_data" / "adult.csv"
output_path = ROOT / "discretized_data" / "adult.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

discretize_preprocess(str(dataset_path), str(output_path), bins=10, strategy='uniform')

Preprocessing: /home/adity/github/katabatic-mentorship-repo/raw_data/adult.csv
Saved preprocessed discrete dataset to: /home/adity/github/katabatic-mentorship-repo/discretized_data/adult.csv


In [3]:
import torch
from katabatic.pipeline.train_test_split.pipeline import TrainTestSplitPipeline
from katabatic.models.vfae_alex.adapter import KatabaticVFAE

# Device Config
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# Set paths
input_csv = str(output_path)
output_dir = str(ROOT / "sample_data" / "adult")
real_test_dir = output_dir
synthetic_dir = str(ROOT / "synthetic" / "adult" / "vfae")

# select protected and target attributes
protected_col = input("Protected Attribute (S): ").strip() or "sex"
target_col = input("Target Attribute (Y): ").strip() or "class"

# VFAE parameters
model_config = {
    "epochs": 50,
    "batch_size": 64,
    "z_dim": 50,         # Latent dimension size
    
    # Fairness Config
    "fairness_config": {
        "S": protected_col,       # Sensitive Column Name
        "Y": target_col,          # Target Column Name
        "S_under": "0",           # Value representing unprivileged group (e.g., '0' for Female)
        "Y_desire": "1"           # Value representing positive outcome (optional context)
    }
}

pipeline = TrainTestSplitPipeline(model=KatabaticVFAE)

pipeline.run(
    input_csv=input_csv,
    output_dir=output_dir,
    synthetic_dir=synthetic_dir,
    real_test_dir=real_test_dir,
    **model_config
)

/home/adity/github/katabatic-mentorship-repo/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded data with shape: (32561, 15)
Saved train/test full data
Train size: (26048, 15), Test size: (6513, 15)
Train label distribution:
 class
0    0.759175
1    0.240825
Name: proportion, dtype: float64
Test label distribution:
 class
0    0.759251
1    0.240749
Name: proportion, dtype: float64
Saved X/y split
Training shape: (26048, 14) (26048,)
Test shape: (6513, 14) (6513,)
Loading VFAE training data from: /home/adity/github/katabatic-mentorship-repo/sample_data/adult
Initialising VFAE (X:13, S:1, Y:1)...


Training VFAE:   0%|          | 0/50 [00:00<?, ?it/s]/home/adity/github/katabatic-mentorship-repo/katabatic/models/vfae_alex/utils.py:16: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([64, 2])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  sup_loss = F.mse_loss(outputs['y_decoded'], y_target, reduction='sum')
Training VFAE: 100%|██████████| 50/50 [01:24<00:00,  1.70s/it, loss=3641.0271]


Generating synthetic data to: /home/adity/github/katabatic-mentorship-repo/synthetic/adult/vfae
Saved artifacts: x_synth.csv ((26048, 14)), y_synth.csv ((26048,))

Results saved to: Results/adult/vfae_tstr.csv

TSTR Evaluation Results:

LR:
Accuracy: 0.6395
F1 Score: 0.6655
AUC: 0.7133

MLP:
Accuracy: 0.6641
F1 Score: 0.6883
AUC: 0.7542

RF:
Accuracy: 0.5867
F1 Score: 0.6088
AUC: 0.7949

XGBoost:
Accuracy: 0.6378
F1 Score: 0.6632
AUC: 0.7646


'Train test split pipeline executed successfully.'